# 07 地震 · 分板块立体时空图

本阶段只用本地主样本，按 Bird (2003) PB2002 板块多边形把每场地震归入所在板块，为**每个有事件的板块各做一张立体图**：板块地图斜铺在底部，每场地震在震中立一根细柱，柱高代表震级。配色为高级紫金。图写入 `figures/3d/`，**不覆盖**已有平面时空图、旧升级图和精细清洗结果。不使用 USGS 目录、活动断层或 CMT，也不开始国家统计局转录或机器学习。


## STEP 1 · 路径

METHOD: 用 `pathlib.Path` 定位 `07_analysis`。优先桌面项目文件夹，若不存在则回退到当前工作目录或箱子上的同名目录。主样本为 `07_earthquake_analysis_sample.csv`。底图在上一级 `mapdata/` 中查找（Natural Earth 50m 陆地）。板块多边形使用已收集的 Bird PB2002 数据，在 `07_external/context/tectonicplates/` 中查找。


In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patheffects as pe
import matplotlib.font_manager as fm
from matplotlib.colors import to_rgba
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
from pyproj import Transformer
from shapely.geometry import box, LineString, Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union
from shapely.validation import make_valid

CANDIDATES = [
    Path(r"C:\Users\WHB\Desktop\DPW I\07_analysis"),
    Path.cwd(),
    Path("/workspace/07_analysis"),
]
ROOT = None
for cand in CANDIDATES:
    if (cand / "07_earthquake_analysis_sample.csv").exists():
        ROOT = cand
        break
if ROOT is None:
    raise FileNotFoundError("找不到 07_earthquake_analysis_sample.csv")

MAP_CANDIDATES = [
    ROOT.parent / "mapdata",
    Path(r"C:\Users\WHB\Desktop\DPW I\mapdata"),
    Path("/workspace/dpw1_report/mapdata"),
]
MAPDATA = next(
    (p for p in MAP_CANDIDATES if (p / "ne_50m_land" / "ne_50m_land.shp").exists()),
    None,
)
if MAPDATA is None:
    raise FileNotFoundError("找不到 Natural Earth 50m 陆地底图 mapdata/ne_50m_land")

PLATE_CANDIDATES = [
    ROOT.parent / "07_external" / "context" / "tectonicplates",
    Path(r"C:\Users\WHB\Desktop\DPW I\07_external\context\tectonicplates"),
    Path("/workspace/07_external/context/tectonicplates"),
]
PLATE_DIR = next(
    (p for p in PLATE_CANDIDATES if (p / "PB2002_plates.shp").exists() or (p / "GeoJSON" / "PB2002_plates.json").exists()),
    None,
)
if PLATE_DIR is None:
    raise FileNotFoundError("找不到 PB2002 板块多边形")
PLATE_SHP = PLATE_DIR / "PB2002_plates.shp"
if not PLATE_SHP.exists():
    PLATE_SHP = PLATE_DIR / "GeoJSON" / "PB2002_plates.json"

SAMPLE_IN = ROOT / "07_earthquake_analysis_sample.csv"
FIGURES = ROOT / "figures" / "3d"
FIGURES.mkdir(parents=True, exist_ok=True)
COUNTS_CSV = ROOT / "07_st_3d_plate_counts.csv"


## STEP 2 · 读取主样本

METHOD: 只读本地主样本。`Date_parsed` 按 UTC、ISO8601 解析。不删行。后续立体图使用坐标有效、经纬度非缺失的事件；主样本为 23,232 条构造地震（1965–2016，M 5.5–9.1）。


In [ ]:
def coerce_bool(s):
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)
    return s.astype(str).str.lower().isin(["true", "1", "yes"])


def load_sample():
    df = pd.read_csv(SAMPLE_IN)
    n0 = len(df)
    df["Date_parsed"] = pd.to_datetime(df["Date_parsed"], format="ISO8601", utc=True, errors="coerce")
    for col in ["flag_coord_invalid", "flag_mag_outlier", "flag_depth_negative", "in_china"]:
        if col in df.columns:
            df[col] = coerce_bool(df[col])
    print("读取主样本:", n0, "Date_parsed 失败:", int(df["Date_parsed"].isna().sum()))
    return df


## STEP 3 · 载入板块多边形并归属

METHOD: 读取 PB2002 板块多边形（shapefile，回退 GeoJSON），先 `make_valid`，再按 `PlateName` 合并被日界线切开的同一板块（如克马德克、巴尔莫勒尔礁）。用空间索引做点在多边形内（`sjoin`，`predicate="within"`）归属；落在边界上或缝隙中的事件，再用 `sjoin_nearest` 归入最近板块。本步只做归属，不修改主样本文件。


In [ ]:
def as_poly(g):
    if g is None or g.is_empty:
        return g
    g = make_valid(g)
    if g.geom_type in ("Polygon", "MultiPolygon"):
        return g
    if g.geom_type == "GeometryCollection":
        polys = [x for x in g.geoms if x.geom_type in ("Polygon", "MultiPolygon") and not x.is_empty]
        return unary_union(polys) if polys else g
    return g


def load_plates():
    plates = gpd.read_file(PLATE_SHP)
    if plates.crs is None:
        plates = plates.set_crs(4326)
    elif plates.crs.to_epsg() != 4326:
        plates = plates.to_crs(4326)
    plates["geometry"] = plates.geometry.map(as_poly)
    # dissolve split parts (Kermadec, Balmoral Reef)
    plates = plates.dissolve(by="PlateName", as_index=False)
    # if Code exists after dissolve it may become a list; keep first
    if "Code" in plates.columns:
        plates["Code"] = plates["Code"].map(lambda x: x if isinstance(x, str) else (x.iloc[0] if hasattr(x, "iloc") else x))
    plates["geometry"] = plates.geometry.map(as_poly)
    plates = plates[plates.geometry.notna() & ~plates.geometry.is_empty].copy()
    plates = plates.reset_index(drop=True)
    print("板块数据:", PLATE_SHP, "要素数:", len(plates))
    return plates


def assign_plates(df, plates):
    spatial = df.loc[
        ~df["flag_coord_invalid"] & df["Latitude"].notna() & df["Longitude"].notna()
    ].copy()
    g = gpd.GeoDataFrame(
        spatial,
        geometry=gpd.points_from_xy(spatial["Longitude"], spatial["Latitude"]),
        crs="EPSG:4326",
    )
    plate_cols = [c for c in ["PlateName", "Code", "geometry"] if c in plates.columns]
    joined = gpd.sjoin(g, plates[plate_cols], how="left", predicate="within")
    # sjoin can duplicate if overlaps exist; keep first
    if not joined.index.is_unique:
        joined = joined[~joined.index.duplicated(keep="first")]
    joined["assign_rule"] = np.where(joined["PlateName"].notna(), "within", pd.NA)
    missing = joined["PlateName"].isna()
    n_miss = int(missing.sum())
    if n_miss:
        leftover = joined.loc[missing, g.columns].copy()
        leftover = leftover.drop(columns=[c for c in leftover.columns if c in ("PlateName", "Code", "index_right", "assign_rule") and c != "geometry"], errors="ignore")
        # rebuild clean leftover from spatial
        leftover = g.loc[missing].copy()
        near = gpd.sjoin_nearest(leftover, plates[plate_cols], how="left", distance_col="plate_dist")
        if not near.index.is_unique:
            near = near[~near.index.duplicated(keep="first")]
        joined.loc[missing, "PlateName"] = near["PlateName"]
        if "Code" in near.columns:
            joined.loc[missing, "Code"] = near["Code"]
        joined.loc[missing, "assign_rule"] = "nearest"
        print("边界/缝隙就近归属:", int(joined["assign_rule"].eq("nearest").sum()))
    n_unassigned = int(joined["PlateName"].isna().sum())
    print("空间样本:", len(joined), "已归属:", len(joined) - n_unassigned, "未归属:", n_unassigned)
    print("归属规则: 点落在板块多边形内则归该板块；在边界上或缝隙中则归最近板块。")
    return joined, n_unassigned


def plate_counts(eq):
    rows = []
    for name, sub in eq.groupby("PlateName", dropna=False):
        rows.append({
            "plate_name": name if pd.notna(name) else "未归属",
            "plate_name_zh": PLATE_CN.get(name, "未归属") if pd.notna(name) else "未归属",
            "code": sub["Code"].dropna().iloc[0] if "Code" in sub.columns and sub["Code"].notna().any() else "",
            "n_events": int(len(sub)),
            "mag_min": float(sub["Magnitude"].min()) if len(sub) else np.nan,
            "mag_max": float(sub["Magnitude"].max()) if len(sub) else np.nan,
            "n_m7": int((sub["Magnitude"] >= 7).sum()) if len(sub) else 0,
            "year_min": int(sub["Date_parsed"].dt.year.min()) if sub["Date_parsed"].notna().any() else "",
            "year_max": int(sub["Date_parsed"].dt.year.max()) if sub["Date_parsed"].notna().any() else "",
            "n_within": int((sub["assign_rule"] == "within").sum()) if "assign_rule" in sub.columns else len(sub),
            "n_nearest": int((sub["assign_rule"] == "nearest").sum()) if "assign_rule" in sub.columns else 0,
        })
    out = pd.DataFrame(rows).sort_values(["n_events", "plate_name"], ascending=[False, True])
    return out


## STEP 4 · 配色、字体与立体投影函数

METHOD: 配色为高级紫金。底图/洋面为近黑深紫，陆地为暗紫褐，柱子由浅金到亮金随震级加深，描边、板块轮廓和经纬线为暗金，文字为暖奶油金。中文字体在 Linux 上用 Noto Sans CJK，在 Windows 上用微软雅黑。每个板块单独选择中央经线，并以等距方位、Equal Earth 或南极立体投影把地图放到局部平面上，再在三维坐标中斜铺；跨 180° 的板块对顶点做逐点投影并在过长跳变处断开，避免日界线撕裂。柱高为 $h = M - 5.0$，各图把 M 9.1 的高度设为图幅较长边的 42%。视角仰角约 30°，方位角按板块长轴自动选择，个别板块可覆盖。


In [ ]:
OCEAN = "#0A0612"
VOID = "#07040D"
PLATE_FILL = "#1C1228"
PLATE_WALL = "#140C1C"
LAND = "#4A2C40"
LAND_EDGE = "#8B6B2E"
GOLD = "#A6843C"
GOLD_DARK = "#8B6B2E"
GOLD_SOFT = "#C4A05A"
CREAM = "#F8F0DC"
MUTED = "#E8DCC0"
BAR_LO = "#E8C98A"
BAR_MID = "#FFD060"
BAR_HI = "#FFE9A0"

GOLD_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "mag_gold", [BAR_LO, BAR_MID, BAR_HI]
)
MAG_NORM = mcolors.Normalize(vmin=5.5, vmax=9.1)
H_OFFSET = 5.0  # h = M - 5.0
H_REF = 9.1 - H_OFFSET  # 4.1, height assigned to M 9.1
Z_FRAC = 0.42  # M 9.1 is 42% of the plate's longer side
SLAB_FRAC = 0.10  # plate slab thickness vs M 9.1 height

def find_cjk_fonts():
    regular_cands = [
        Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"),
        Path(r"C:\Windows\Fonts\msyh.ttc"),
        Path(r"C:\Windows\Fonts\msyh.ttf"),
        Path(r"C:\Windows\Fonts\simhei.ttf"),
        Path(r"C:\Windows\Fonts\simsun.ttc"),
    ]
    bold_cands = [
        Path("/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc"),
        Path(r"C:\Windows\Fonts\msyhbd.ttc"),
        Path(r"C:\Windows\Fonts\msyhbd.ttf"),
    ]
    regular = next((p for p in regular_cands if p.exists()), None)
    bold = next((p for p in bold_cands if p.exists()), regular)
    if regular is None:
        for name in ["Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "Source Han Sans SC"]:
            for f in fm.fontManager.ttflist:
                if name.lower() in f.name.lower():
                    regular = Path(f.fname)
                    break
            if regular:
                break
    if regular is None:
        raise RuntimeError("未找到中文字体")
    return (
        fm.FontProperties(fname=str(regular)),
        fm.FontProperties(fname=str(bold if bold else regular)),
    )


FONT, FONT_BOLD = find_cjk_fonts()
HALO = [pe.withStroke(linewidth=2.6, foreground=OCEAN)]

PLATE_CN = {
    "Africa": "非洲板块",
    "Antarctica": "南极板块",
    "Somalia": "索马里板块",
    "India": "印度板块",
    "Australia": "澳大利亚板块",
    "Eurasia": "欧亚板块",
    "North America": "北美板块",
    "South America": "南美板块",
    "Nazca": "纳斯卡板块",
    "Pacific": "太平洋板块",
    "Arabia": "阿拉伯板块",
    "Sunda": "巽他板块",
    "Timor": "帝汶板块",
    "Kermadec": "克马德克板块",
    "Tonga": "汤加板块",
    "Niuafo'ou": "纽阿福欧板块",
    "Woodlark": "伍德拉克板块",
    "Maoke": "毛克板块",
    "South Bismarck": "南俾斯麦板块",
    "Solomon Sea": "所罗门海板块",
    "North Bismarck": "北俾斯麦板块",
    "New Hebrides": "新赫布里底板块",
    "Caribbean": "加勒比板块",
    "Cocos": "科科斯板块",
    "Okhotsk": "鄂霍次克板块",
    "Juan de Fuca": "胡安·德富卡板块",
    "Altiplano": "阿尔蒂普拉诺板块",
    "North Andes": "北安第斯板块",
    "Okinawa": "冲绳板块",
    "Philippine Sea": "菲律宾海板块",
    "Amur": "阿穆尔板块",
    "Caroline": "加罗林板块",
    "Mariana": "马里亚纳板块",
    "Futuna": "富图纳板块",
    "Scotia": "斯科舍板块",
    "Shetland": "南设得兰板块",
    "Aegean Sea": "爱琴海板块",
    "Anatolia": "安纳托利亚板块",
    "Yangtze": "扬子板块",
    "Burma": "缅甸板块",
    "Rivera": "里维拉板块",
    "Birds Head": "鸟头板块",
    "Molucca Sea": "马鲁古海板块",
    "Banda Sea": "班达海板块",
    "Manus": "马努斯板块",
    "Conway Reef": "康威礁板块",
    "Balmoral Reef": "巴尔莫勒尔礁板块",
    "Easter": "复活节岛板块",
    "Juan Fernandez": "胡安·费尔南德斯板块",
    "Galapagos": "加拉帕戈斯板块",
    "Sandwich": "桑威奇板块",
    "Panama": "巴拿马板块",
}

# Manual view tweaks filled in after visual QA (elev, azim)
VIEW_OVERRIDE = {
    # name: (elev, azim) — only when the auto view is poor
    "Antarctica": (32, 15),
    "Pacific": (30, -70),
    "North America": (30, -70),
    "Eurasia": (30, -70),
    "Australia": (30, -70),
    "South America": (30, -40),
    "Africa": (30, -55),
    "Nazca": (30, -55),
    "Kermadec": (30, -15),
    "Tonga": (30, -15),
}

def as_poly(g):
    if g is None or g.is_empty:
        return g
    g = make_valid(g)
    if g.geom_type in ("Polygon", "MultiPolygon"):
        return g
    if g.geom_type == "GeometryCollection":
        polys = [x for x in g.geoms if x.geom_type in ("Polygon", "MultiPolygon") and not x.is_empty]
        return unary_union(polys) if polys else g
    return g


def iter_polys(g):
    if g is None or g.is_empty:
        return
    g = as_poly(g)
    if g is None or g.is_empty:
        return
    if g.geom_type == "Polygon":
        yield g
    elif g.geom_type == "MultiPolygon":
        for p in g.geoms:
            if not p.is_empty:
                yield p


def collect_coords(g):
    arrs = []
    for p in iter_polys(g):
        arrs.append(np.asarray(p.exterior.coords))
    if not arrs:
        return np.empty((0, 2))
    return np.vstack(arrs)


def sph_centroid(lons, lats):
    lon = np.deg2rad(np.asarray(lons, dtype=float))
    lat = np.deg2rad(np.asarray(lats, dtype=float))
    x = np.cos(lat) * np.cos(lon)
    y = np.cos(lat) * np.sin(lon)
    z = np.sin(lat)
    x, y, z = x.mean(), y.mean(), z.mean()
    lon0 = float(np.rad2deg(np.arctan2(y, x)))
    lat0 = float(np.rad2deg(np.arctan2(z, np.hypot(x, y))))
    return lon0, lat0


def wrap_lon(lon):
    return ((np.asarray(lon, dtype=float) + 180.0) % 360.0) - 180.0


def unwrap_lons(lons, lon0):
    return wrap_lon(np.asarray(lons, dtype=float) - lon0) + lon0



def plate_center_and_proj(geom):
    arr = collect_coords(geom)
    if len(arr) == 0:
        return 0.0, 0.0, "aeqd"
    lon_c, lat_c = sph_centroid(arr[:, 0], arr[:, 1])
    ul = unwrap_lons(arr[:, 0], lon_c)
    lon0 = float(0.5 * (ul.min() + ul.max()))
    lon0 = float(wrap_lon(lon0))
    lat_min, lat_max = float(arr[:, 1].min()), float(arr[:, 1].max())
    lat0 = float(0.5 * (lat_min + lat_max))
    lon_span = float(ul.max() - ul.min())
    lat_span = lat_max - lat_min
    # south polar plate
    if lat_min <= -89.5 and lat_span > 30:
        return lon0, -90.0, "stere_s"
    # north polar (NA): keep a mid-latitude centre, use eqearth
    if lat_max >= 89.5:
        # 北美板块包含北极：用中高纬等距方位投影，避免等积地球投影在极区拉裂
        return lon0, 65.0, "aeqd"
    if lon_span >= 70 or lat_span >= 50:
        return lon0, lat0, "eqearth"
    return lon0, lat0, "aeqd"


def crs_for(lon0, lat0, kind):
    if kind == "stere_s":
        return f"+proj=stere +lat_0=-90 +lat_ts=-71 +lon_0={lon0:.6f} +datum=WGS84 +units=m +no_defs"
    if kind == "eqearth":
        return f"+proj=eqearth +lon_0={lon0:.6f} +datum=WGS84 +units=m +no_defs"
    return f"+proj=aeqd +lat_0={lat0:.6f} +lon_0={lon0:.6f} +R=6371000 +units=m +no_defs"


def make_graticule(lon_min, lon_max, lat_min, lat_max, dlon=10, dlat=10):
    # work in unwrapped lon space, then wrap back
    lines = []
    lon0 = 0.5 * (lon_min + lon_max)
    # meridians
    start = np.floor(lon_min / dlon) * dlon
    end = np.ceil(lon_max / dlon) * dlon
    lats = np.arange(max(-89.5, lat_min - dlat), min(89.5, lat_max + dlat) + 0.01, 1.0)
    for lon in np.arange(start, end + 0.01, dlon):
        lines.append(LineString([(wrap_lon(lon), la) for la in lats]))
    # parallels
    lon_s = np.linspace(lon_min, lon_max, max(20, int((lon_max - lon_min) / 1.5)))
    lat_start = np.floor(lat_min / dlat) * dlat
    lat_end = np.ceil(lat_max / dlat) * dlat
    for lat in np.arange(lat_start, lat_end + 0.01, dlat):
        if abs(lat) >= 89.9:
            continue
        lines.append(LineString([(wrap_lon(lo), lat) for lo in lon_s]))
    return gpd.GeoDataFrame(geometry=lines, crs="EPSG:4326")


def grat_step(lon_span, lat_span):
    span = max(lon_span, lat_span)
    if span >= 80:
        return 20, 20
    if span >= 35:
        return 10, 10
    if span >= 15:
        return 5, 5
    if span >= 6:
        return 2, 2
    return 1, 1


def slugify(name: str) -> str:
    s = name.lower().replace(" ", "_").replace("'", "")
    s = re.sub(r"[^a-z0-9_]+", "", s)
    return s


def wrap_cjk(text, width=78):
    lines, buf = [], ""
    for ch in text:
        buf += ch
        if len(buf) >= width and ch in "。；，、： ":
            lines.append(buf.strip())
            buf = ""
    if buf.strip():
        lines.append(buf.strip())
    return "\n".join(lines)


def mag_height(mag):
    return np.asarray(mag, dtype=float) - H_OFFSET


def bar_style(n, short_span):
    if n >= 1200:
        frac, alpha = 0.0032, 0.58
    elif n >= 600:
        frac, alpha = 0.0042, 0.65
    elif n >= 200:
        frac, alpha = 0.0055, 0.76
    elif n >= 40:
        frac, alpha = 0.0080, 0.88
    else:
        frac, alpha = 0.0120, 0.95
    dx = short_span * frac
    return dx, alpha


def proj_label(kind, lon0, lat0):
    if kind == "stere_s":
        return f"南极立体投影（中央经线 {lon0:.1f}°）"
    if kind == "eqearth":
        return f"Equal Earth 等积投影（中央经线 {lon0:.1f}°）"
    hemi = "N" if lat0 >= 0 else "S"
    return f"等距方位投影（中央经线 {lon0:.1f}°，中央纬线 {abs(lat0):.1f}°{hemi}）"

# ---------------------------------------------------------------------------
# Pointwise projection (avoids geopandas antimeridian / polar tears)
# ---------------------------------------------------------------------------
def _transformer(crs):
    return Transformer.from_crs("EPSG:4326", crs, always_xy=True)


def project_xy(lons, lats, crs):
    tf = _transformer(crs)
    x, y = tf.transform(np.asarray(lons, dtype=float), np.asarray(lats, dtype=float))
    return np.asarray(x, dtype=float), np.asarray(y, dtype=float)


def _dedup_ring(x, y, eps=1e-6):
    if len(x) == 0:
        return x, y
    keep = [0]
    for i in range(1, len(x)):
        if abs(x[i] - x[keep[-1]]) > eps or abs(y[i] - y[keep[-1]]) > eps:
            keep.append(i)
    # keep last if it closed the ring to the first
    if len(keep) >= 2 and (abs(x[keep[-1]] - x[keep[0]]) > eps or abs(y[keep[-1]] - y[keep[0]]) > eps):
        pass
    return np.asarray(x)[keep], np.asarray(y)[keep]


def _split_on_jumps(x, y, thresh):
    """Split a ring into segments wherever consecutive vertices jump too far."""
    if len(x) < 2:
        return [(x, y)] if len(x) else []
    parts = []
    sx, sy = [x[0]], [y[0]]
    for i in range(1, len(x)):
        d = float(np.hypot(x[i] - x[i - 1], y[i] - y[i - 1]))
        if d > thresh:
            if len(sx) >= 2:
                parts.append((np.asarray(sx), np.asarray(sy)))
            sx, sy = [x[i]], [y[i]]
        else:
            sx.append(x[i])
            sy.append(y[i])
    if len(sx) >= 2:
        parts.append((np.asarray(sx), np.asarray(sy)))
    return parts


def project_rings(geom, crs, jump_thresh=None):
    """Return list of (x, y) exterior rings in projected metres."""
    rings = []
    tf = _transformer(crs)
    for p in iter_polys(geom):
        # densify so projected edges curve correctly
        try:
            p = p.segmentize(0.4)
        except Exception:
            pass
        coords = np.asarray(p.exterior.coords)
        x, y = tf.transform(coords[:, 0], coords[:, 1])
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        ok = np.isfinite(x) & np.isfinite(y)
        x, y = x[ok], y[ok]
        x, y = _dedup_ring(x, y)
        if len(x) < 3:
            continue
        if jump_thresh is not None:
            rings.extend(_split_on_jumps(x, y, jump_thresh))
        else:
            rings.append((x, y))
    return rings


def rings_to_faces(rings, z=0.0):
    faces = []
    for x, y in rings:
        if len(x) < 3:
            continue
        faces.append(list(zip(x, y, np.full(len(x), z, dtype=float))))
    return faces


def rings_to_wall_faces(rings, z_top, z_bot):
    faces = []
    for x, y in rings:
        n = len(x)
        if n < 2:
            continue
        for i in range(n - 1):
            faces.append([
                (x[i], y[i], z_top),
                (x[i + 1], y[i + 1], z_top),
                (x[i + 1], y[i + 1], z_bot),
                (x[i], y[i], z_bot),
            ])
    return faces


def rings_to_lines(rings, z=0.0):
    segs = []
    for x, y in rings:
        if len(x) < 2:
            continue
        segs.append(np.column_stack([x, y, np.full(len(x), z, dtype=float)]))
    return segs


def robust_span(xs, ys):
    x = np.concatenate(xs) if xs else np.array([])
    y = np.concatenate(ys) if ys else np.array([])
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 3:
        return 0, 1, 0, 1
    # drop extreme outliers (projection singularities)
    lo, hi = 0.002, 0.998
    xmin, xmax = np.quantile(x, [lo, hi])
    ymin, ymax = np.quantile(y, [lo, hi])
    # but if almost all points are well clustered, use min/max of inliers
    inlier = (x >= xmin - 0.05 * (xmax - xmin + 1)) & (x <= xmax + 0.05 * (xmax - xmin + 1))
    inlier &= (y >= ymin - 0.05 * (ymax - ymin + 1)) & (y <= ymax + 0.05 * (ymax - ymin + 1))
    if inlier.sum() >= 3:
        return float(x[inlier].min()), float(x[inlier].max()), float(y[inlier].min()), float(y[inlier].max())
    return float(xmin), float(xmax), float(ymin), float(ymax)


def project_lines(geoms, crs, jump_thresh):
    segs = []
    tf = _transformer(crs)

    def add_line(g):
        if g is None or g.is_empty:
            return
        if g.geom_type == "MultiLineString":
            for gg in g.geoms:
                add_line(gg)
            return
        if g.geom_type != "LineString":
            return
        try:
            g = g.segmentize(0.5)
        except Exception:
            pass
        c = np.asarray(g.coords)
        x, y = tf.transform(c[:, 0], c[:, 1])
        x = np.asarray(x, dtype=float)
        y = np.asarray(y, dtype=float)
        ok = np.isfinite(x) & np.isfinite(y)
        x, y = x[ok], y[ok]
        for px, py in _split_on_jumps(x, y, jump_thresh):
            if len(px) >= 2:
                segs.append((px, py))
    for g in geoms:
        add_line(g)
    return segs


def auto_azim(xmin, xmax, ymin, ymax):
    xr = max(xmax - xmin, 1.0)
    yr = max(ymax - ymin, 1.0)
    if yr > 1.35 * xr:
        return -18  # look from the east so a N–S plate runs across the frame
    if xr > 1.8 * yr:
        return -75  # look from the SSW so an E–W plate runs across
    return -60


# ---------------------------------------------------------------------------
# 3D figure
# ---------------------------------------------------------------------------
def style_3d_ax(ax):
    ax.set_facecolor(VOID)
    # hide the default 3D box / grid completely
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        try:
            axis.pane.set_visible(False)
        except Exception:
            pass
        axis.pane.set_facecolor((0, 0, 0, 0))
        axis.pane.set_edgecolor((0, 0, 0, 0))
        axis.pane.set_alpha(0)
        axis.pane.fill = False
        axis.line.set_color((0, 0, 0, 0))
        axis._axinfo["grid"]["color"] = (0, 0, 0, 0)
        axis._axinfo["grid"]["linewidth"] = 0
        axis._axinfo["tick"]["inward_factor"] = 0
        axis._axinfo["tick"]["outward_factor"] = 0
        axis._axinfo["axisline"]["color"] = (0, 0, 0, 0)
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_zlabel("")
    ax.set_axis_off()


def render_plate(name, plate_row, events, land, out_path, dpi=300):
    geom = as_poly(plate_row.geometry)
    try:
        geom = geom.segmentize(0.35)
    except Exception:
        pass
    lon0, lat0, kind = plate_center_and_proj(geom)
    crs = crs_for(lon0, lat0, kind)
    arr = collect_coords(geom)
    ul = unwrap_lons(arr[:, 0], lon0)
    lon_span = float(ul.max() - ul.min())
    lat_span = float(arr[:, 1].max() - arr[:, 1].min())

    plate_gdf = gpd.GeoDataFrame([{"PlateName": name, "geometry": geom}], crs="EPSG:4326")

    # rough projected rings to set a jump threshold
    raw_rings = project_rings(geom, crs, jump_thresh=None)
    if not raw_rings:
        print("WARNING: no projected rings for", name)
        return None
    xmin, xmax, ymin, ymax = robust_span([r[0] for r in raw_rings], [r[1] for r in raw_rings])
    long_span = max(xmax - xmin, ymax - ymin, 1.0)
    jump_thresh = 0.35 * long_span
    plate_rings = project_rings(geom, crs, jump_thresh=jump_thresh)
    if not plate_rings:
        plate_rings = raw_rings
    xmin, xmax, ymin, ymax = robust_span([r[0] for r in plate_rings], [r[1] for r in plate_rings])
    xr = max(xmax - xmin, 1.0)
    yr = max(ymax - ymin, 1.0)
    long_span = max(xr, yr)
    short_span = min(xr, yr)

    # land clipped to the plate in geographic space, then projected pointwise
    try:
        land_clip = gpd.clip(land, plate_gdf)
    except Exception:
        land_clip = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    land_clip = land_clip[land_clip.geometry.notna() & ~land_clip.geometry.is_empty]
    land_rings = []
    if len(land_clip):
        for g in land_clip.geometry:
            land_rings.extend(project_rings(g, crs, jump_thresh=jump_thresh))

    # graticule clipped to plate
    dlon, dlat = grat_step(lon_span, lat_span)
    grat = make_graticule(ul.min(), ul.max(), arr[:, 1].min(), arr[:, 1].max(), dlon, dlat)
    try:
        grat = gpd.clip(grat, plate_gdf)
    except Exception:
        pass
    grat_segs = []
    if len(grat):
        grat_segs = project_lines(list(grat.geometry), crs, jump_thresh)

    n = len(events)
    xs, ys = project_xy(events["Longitude"].to_numpy(), events["Latitude"].to_numpy(), crs)
    mags = events["Magnitude"].to_numpy(dtype=float)
    ok = np.isfinite(xs) & np.isfinite(ys)
    # keep points that fall inside the plate's projected bounds (plus a little pad)
    pad_ok = (xs >= xmin - 0.08 * xr) & (xs <= xmax + 0.08 * xr) & (ys >= ymin - 0.08 * yr) & (ys <= ymax + 0.08 * yr)
    ok = ok & pad_ok
    xs, ys, mags = xs[ok], ys[ok], mags[ok]
    n = len(mags)
    order = np.argsort(mags)
    xs, ys, mags = xs[order], ys[order], mags[order]
    hs = mag_height(mags)
    colors = GOLD_CMAP(MAG_NORM(mags))

    z_visual = Z_FRAC * long_span
    zmax_data = H_REF * 1.08
    slab = SLAB_FRAC * H_REF  # thickness in data z units
    dx, alpha = bar_style(n, short_span)
    if n:
        colors = colors.copy()
        colors[:, 3] = alpha

    fig = plt.figure(figsize=(13.0, 9.4), facecolor=VOID)
    ax = fig.add_subplot(111, projection="3d")
    style_3d_ax(ax)
    fig.subplots_adjust(left=0.02, right=0.86, top=0.90, bottom=0.12)
    try:
        ax.set_proj_type("persp", focal_length=0.9)
    except TypeError:
        ax.set_proj_type("persp")

    # slab underside + walls (slightly below z=0) then plate floor at z=0
    z_bot = -slab
    wall_faces = rings_to_wall_faces(plate_rings, 0.0, z_bot)
    if wall_faces:
        walls = Poly3DCollection(
            wall_faces, facecolors=PLATE_WALL, edgecolors="none", linewidths=0, zorder=1,
        )
        walls.set_sort_zpos(-0.5)
        ax.add_collection3d(walls)
    bot_faces = rings_to_faces(plate_rings, z=z_bot)
    if bot_faces:
        bot = Poly3DCollection(
            bot_faces, facecolors=PLATE_WALL, edgecolors="none", linewidths=0, zorder=0,
        )
        bot.set_sort_zpos(-1.0)
        ax.add_collection3d(bot)

    ocean_faces = rings_to_faces(plate_rings, z=0.0)
    if ocean_faces:
        ocean_col = Poly3DCollection(
            ocean_faces, facecolors=PLATE_FILL, edgecolors="none", linewidths=0, zorder=2,
        )
        ocean_col.set_sort_zpos(0.0)
        ax.add_collection3d(ocean_col)

    if land_rings:
        land_faces = rings_to_faces(land_rings, z=0.0)
        if land_faces:
            land_col = Poly3DCollection(
                land_faces,
                facecolors=LAND,
                edgecolors=to_rgba(LAND_EDGE, 0.55),
                linewidths=0.22,
                zorder=3,
            )
            land_col.set_sort_zpos(0.15)
            ax.add_collection3d(land_col)

    if grat_segs:
        g_lines = [
            np.column_stack([x, y, np.full(len(x), 0.0)]) for x, y in grat_segs if len(x) >= 2
        ]
        if g_lines:
            lc = Line3DCollection(g_lines, colors=to_rgba(GOLD, 0.28), linewidths=0.35, zorder=4)
            lc.set_sort_zpos(0.2)
            ax.add_collection3d(lc)

    outline = rings_to_lines(plate_rings, z=0.0)
    if outline:
        oc = Line3DCollection(outline, colors=to_rgba("#C4A05A", 0.98), linewidths=1.55, zorder=5)
        oc.set_sort_zpos(0.35)
        ax.add_collection3d(oc)
        # bottom gold edge of the slab
        oc2 = Line3DCollection(
            rings_to_lines(plate_rings, z=z_bot),
            colors=to_rgba(GOLD_DARK, 0.7), linewidths=0.7, zorder=1,
        )
        oc2.set_sort_zpos(-0.2)
        ax.add_collection3d(oc2)

    if n:
        ax.bar3d(
            xs - dx / 2.0,
            ys - dx / 2.0,
            np.zeros_like(hs),
            dx,
            dx,
            hs,
            color=colors,
            shade=True,
            zsort="average",
        )

    pad_x, pad_y = xr * 0.07, yr * 0.09
    ax.set_xlim(xmin - pad_x, xmax + pad_x)
    ax.set_ylim(ymin - pad_y, ymax + pad_y)
    ax.set_zlim(z_bot, zmax_data)
    # visual z includes the slab so the box aspect stays honest
    z_vis_total = z_visual * (zmax_data - z_bot) / zmax_data
    try:
        ax.set_box_aspect((xr + 2 * pad_x, yr + 2 * pad_y, z_vis_total), zoom=1.45)
    except TypeError:
        ax.set_box_aspect((xr + 2 * pad_x, yr + 2 * pad_y, z_vis_total))

    elev_azim = VIEW_OVERRIDE.get(name)
    if elev_azim is None:
        elev, azim = 30, auto_azim(xmin, xmax, ymin, ymax)
    else:
        elev, azim = elev_azim
    ax.view_init(elev=elev, azim=azim)

    cn = PLATE_CN.get(name, name)
    code = ""
    try:
        code = plate_row["Code"]
        if not isinstance(code, str):
            code = ""
    except Exception:
        code = ""
    if code:
        title_s = f"{cn}（{name}, {code}）构造地震立体分布（1965–2016）"
    else:
        title_s = f"{cn}（{name}）构造地震立体分布（1965–2016）"
    fig.suptitle(title_s, fontproperties=FONT_BOLD, fontsize=16, color=CREAM, y=0.975)
    # path effects on suptitle
    if fig._suptitle is not None:
        fig._suptitle.set_path_effects(HALO)

    sm = plt.cm.ScalarMappable(cmap=GOLD_CMAP, norm=MAG_NORM)
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.01, shrink=0.52)
    cb.ax.yaxis.set_tick_params(color=GOLD, labelcolor=CREAM, labelsize=8)
    cb.outline.set_edgecolor(GOLD)
    cb.outline.set_linewidth(0.6)
    cb.set_label("震级", fontproperties=FONT, color=CREAM, fontsize=9, labelpad=8)
    for t in cb.ax.get_yticklabels():
        t.set_fontproperties(FONT)
        t.set_color(CREAM)

    n_m7 = int((mags >= 7).sum()) if n else 0
    mag_min = float(mags.min()) if n else float("nan")
    mag_max = float(mags.max()) if n else float("nan")
    cap = (
        f"数据：本地主样本构造地震（1965–2016，M 5.5–9.1）。"
        f"归属：Bird (2003) PB2002 板块多边形；点在多边形内归入该板块，落在边界上则归最近板块。"
        f"本图 n={n:,}，震级 {mag_min:.1f}–{mag_max:.1f}，其中 M≥7 共 {n_m7} 次。"
        f"每场地震在震中立一根柱，竖直高度 h = M − 5.0；本图将 M 9.1 的高度设为图幅较长边的 {int(Z_FRAC*100)}%，"
        f"因此同一图内 9.1 明显高于 5.5，不同板块图的绝对柱高不可直接比较。"
        f"柱色由浅金到亮金随震级升高。投影：{proj_label(kind, lon0, lat0)}。"
        f"底图：Natural Earth 50m 陆地，已裁到本板块。视角：仰角 {elev}°，方位角 {azim}°。"
    )
    fig.text(
        0.015, 0.012, wrap_cjk(cap, width=76),
        fontproperties=FONT, fontsize=7.4, color=MUTED,
        ha="left", va="bottom", linespacing=1.38,
    )

    fig.savefig(
        out_path, dpi=dpi, facecolor=fig.get_facecolor(), edgecolor="none",
        bbox_inches="tight", pad_inches=0.20,
    )
    plt.close(fig)
    print(
        f"wrote {out_path}  n={n}  proj={kind} lon0={lon0:.1f} lat0={lat0:.1f}  "
        f"view={elev},{azim}  bytes={Path(out_path).stat().st_size}"
    )
    return {
        "name": name, "n": n, "kind": kind, "lon0": lon0, "lat0": lat0,
        "elev": elev, "azim": azim, "path": str(out_path),
    }


## STEP 5 · 导出板块计数表

METHOD: 完成归属后，写出每个板块的事件数、震级最小/最大、M≥7 次数和年份范围，保存为 `07_st_3d_plate_counts.csv`。同时载入 Natural Earth 50m 陆地，供下一步裁切。


In [ ]:
df = load_sample()
plates = load_plates()
eq, n_unassigned = assign_plates(df, plates)
counts = plate_counts(eq)
counts.to_csv(COUNTS_CSV, index=False)
print("wrote", COUNTS_CSV)
print(counts.to_string(index=False))
print("未归属:", n_unassigned)
print("板块数据来源:", PLATE_SHP)

land = gpd.read_file(MAPDATA / "ne_50m_land" / "ne_50m_land.shp")
if land.crs is None:
    land = land.set_crs(4326)
elif land.crs.to_epsg() != 4326:
    land = land.to_crs(4326)
print("陆地底图:", MAPDATA / "ne_50m_land")


## STEP 6 · 逐板块绘制立体图

METHOD: 每个有事件的板块单独出一张图，保存到 `figures/3d/07_st_3d_<plate_slug>.png`，300 dpi。不聚合震中，每场地震一根柱；密区用细柱和透明度。地图是单幅三维场景：板块做成斜铺的薄台，陆地裁到该板块，柱子从震中竖起。不写入已有平面时空图文件名。


In [ ]:
FIGURES_3D = FIGURES
names = list(counts.loc[counts["plate_name"] != "未归属", "plate_name"])
plate_lut = plates.set_index("PlateName")
results = []
PROTECTED_2D = {
    "07_st_annual_mag.png",
    "07_st_decade_mag.png",
    "07_st_global_density_m7.png",
    "07_st_global_depth.png",
}
for name in names:
    slug = slugify(name)
    out_name = f"07_st_3d_{slug}.png"
    if out_name in PROTECTED_2D:
        raise RuntimeError(f"拒绝覆盖受保护的平面图文件名: {out_name}")
    sub = eq.loc[eq["PlateName"] == name]
    row = plate_lut.loc[name]
    if isinstance(row, pd.DataFrame):
        row = row.iloc[0]
    path = FIGURES_3D / out_name
    info = render_plate(name, row, sub, land, path, dpi=300)
    if info:
        results.append(info)
print("完成图数:", len(results))


## STEP 7 · 检查输出，确认旧文件未被覆盖

METHOD: 列出新的立体图和计数表，并确认 `figures/07_st_annual_mag.png` 等平面时空图、旧升级图和既有时空格局笔记本均仍在，且本本没有覆盖它们。


In [ ]:
from datetime import datetime

print("—— 立体图输出检查 ——")
print("笔记本将写入的新文件均在 figures/3d/ 或 07_st_3d_plate_counts.csv")
print("计数表:", COUNTS_CSV.exists(), "bytes:", COUNTS_CSV.stat().st_size if COUNTS_CSV.exists() else 0)
print("立体图目录:", FIGURES_3D, "文件数:", len(list(FIGURES_3D.glob("07_st_3d_*.png"))))
for p in sorted(FIGURES_3D.glob("07_st_3d_*.png")):
    print(f"  {p.name:40s}  {p.stat().st_size:9d}")

# 确认平面图仍在且未被本本写入
fig2d = ROOT / "figures"
protected = [
    fig2d / "07_st_annual_mag.png",
    fig2d / "07_st_decade_mag.png",
    fig2d / "07_st_global_density_m7.png",
    fig2d / "07_st_global_depth.png",
]
print()
print("平面时空图仍在（本本未覆盖）:")
for p in protected:
    print(" ", p.name, "exists:", p.exists(), "bytes:", p.stat().st_size if p.exists() else 0,
          "mtime:", datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds") if p.exists() else "-")

old_maps = list(fig2d.glob("07_map_*.png"))
print("未改动旧升级图 07_map_*.png 数量:", len(old_maps), "(本本不写入这些文件名)")
print("主样本仍在:", SAMPLE_IN.exists())
print("既有时空格局笔记本仍在:", (ROOT / "07_earthquake_analysis_spatiotemporal.ipynb").exists())
print("立体图未写进 figures/ 根目录:", not any((fig2d / f"07_st_3d_{slugify(n)}.png").exists() and (fig2d / f"07_st_3d_{slugify(n)}.png") == (FIGURES_3D / f"07_st_3d_{slugify(n)}.png") for n in []) )
# 3d files live in figures/3d, not figures/
leaked = [p.name for p in fig2d.glob("07_st_3d_*.png") if p.parent.resolve() == fig2d.resolve()]
print("平面图目录根下误写入的 07_st_3d_*.png:", leaked)
print("未归属事件数:", n_unassigned)
